In [ ]:
import numpy as np
from sympy import symbols

from psiop_qwen import *

# --- 1-D Laplacian on [-1, 1] inside [-2, 2] ---
x_sym, xi = symbols('x xi', real=True)
op = PseudoDifferentialOperator(xi**2, [x_sym], mode='symbol')

N = 256
x = np.linspace(-2, 2, N, endpoint=False)
dx = x[1] - x[0]
kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)

g = lambda xv: (xv + 1) * (xv - 1)          # ≤ 0 on [-1, 1]
geo = SubdomainGeometry(x, g, dim=1)
geo.check_resolution()

pipe = SubdomainBoundaryPipeline(op, geo, f=1.0,
                                 max_defect_iters=8, defect_tol=1e-8)
u = np.ones(N, dtype=complex)
v, diag = pipe.apply(u, x, kx)

print(f"residual_D = {abs(diag['residual_D']):.6e}")
print(f"converged  = {diag['converged']}  in {diag['n_iters']} iters")

# --- scope check on a fractional symbol (should FAIL) ---
try:
    bad = PseudoDifferentialOperator(abs(xi)**0.5, [x_sym], mode='symbol')
    SubdomainBoundaryPipeline(bad, geo, f=1.0)
except NotImplementedError as e:
    print(f"Caught expected error: {e}")

# --- validation helper ---
pipe.validate_laplacian_1d(N=256)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, I


# ==============================================================================
# 1D SETUP
# ==============================================================================
N1 = 512
L1 = 3.0
x = np.linspace(-L1, L1, N1, endpoint=False)
dx = x[1] - x[0]
kx = 2 * np.pi * np.fft.fftfreq(N1, d=dx)
x_sym, xi = symbols('x xi', real=True)

# 1.1 Laplacian
op_lap = PseudoDifferentialOperator(xi**2, [x_sym], mode='symbol')
geo_lap = SubdomainGeometry(x, lambda xv: (xv + 1.0) * (xv - 1.0), dim=1)
pipe_lap = SubdomainBoundaryPipeline(op_lap, geo_lap, f=1.0, max_defect_iters=8)
v_lap, diag_lap = pipe_lap.apply(x**2, x, kx)
exact_lap = np.full_like(x, 2.0, dtype=complex)

# 1.2 Transport
op_trans = PseudoDifferentialOperator(I*xi, [x_sym], mode='symbol')
geo_trans = SubdomainGeometry(x, lambda xv: (xv - 0.0) * (xv - 2.0), dim=1)
pipe_trans = SubdomainBoundaryPipeline(op_trans, geo_trans, f=0.0, max_defect_iters=8)
u_trans = np.sin(np.pi * x / 2.0)
v_trans, diag_trans = pipe_trans.apply(u_trans, x, kx)
exact_trans = (np.pi / 2.0) * np.cos(np.pi * x / 2.0)

# 1.3 Bessel Potential (Weakly Nonlocal)
op_bes = PseudoDifferentialOperator(1 / (1 + xi**2), [x_sym], mode='symbol')
geo_bes = SubdomainGeometry(x, lambda xv: (xv + 1.5) * (xv - 1.5), dim=1)
pipe_bes = SubdomainBoundaryPipeline(op_bes, geo_bes, f=np.exp(-1.5**2), max_defect_iters=5)
u_bes = np.exp(-x**2)
v_bes, diag_bes = pipe_bes.apply(u_bes, x, kx)

# ==============================================================================
# 2D SETUP
# ==============================================================================
N2 = 128
L2 = 2.5
x2 = np.linspace(-L2, L2, N2, endpoint=False)
y2 = np.linspace(-L2, L2, N2, endpoint=False)
dx2, dy2 = x2[1] - x2[0], y2[1] - y2[0]
kx2 = 2 * np.pi * np.fft.fftfreq(N2, d=dx2)
ky2 = 2 * np.pi * np.fft.fftfreq(N2, d=dy2)
X2, Y2 = np.meshgrid(x2, y2, indexing='ij')
x_sym, y_sym, xi, eta = symbols('x y xi eta', real=True)

# 2.1 Laplacian on Disk
op_lap2 = PseudoDifferentialOperator(xi**2 + eta**2, [x_sym, y_sym], mode='symbol')
geo_disk = SubdomainGeometry(x2, lambda X, Y: X**2 + Y**2 - 1.0**2, dim=2, y_grid=y2)
pipe_disk = SubdomainBoundaryPipeline(op_lap2, geo_disk, f=1.0, max_defect_iters=8)
v_disk, diag_disk = pipe_disk.apply(X2**2 + Y2**2, x2, kx2, y_grid=y2, ky=ky2)
exact_disk = np.full_like(X2, 4.0, dtype=complex)

# 2.2 Anisotropic on Ellipse
op_aniso = PseudoDifferentialOperator(xi**2 + 4*eta**2, [x_sym, y_sym], mode='symbol')
geo_ell = SubdomainGeometry(x2, lambda X, Y: (X/2.0)**2 + Y**2 - 1.0, dim=2, y_grid=y2)
pipe_ell = SubdomainBoundaryPipeline(op_aniso, geo_ell, f=lambda X, Y: X**2 + Y**2, max_defect_iters=8)
v_ell, diag_ell = pipe_ell.apply(X2**2 + Y2**2, x2, kx2, y_grid=y2, ky=ky2)
exact_ell = np.full_like(X2, 10.0, dtype=complex)

# 2.3 Joint Symbol (Variable Coeff)
# x * d_x + d_y^2  => symbol: x*(I*xi) - eta**2
op_joint = PseudoDifferentialOperator(x_sym*(I*xi) - eta**2, [x_sym, y_sym], 
                                      mode='symbol', apply_backend='peetre', compute_peetre=True)
geo_joint = SubdomainGeometry(x2, lambda X, Y: X**2 + Y**2 - 1.2**2, dim=2, y_grid=y2)
pipe_joint = SubdomainBoundaryPipeline(op_joint, geo_joint, f=lambda X, Y: X * Y**2, max_defect_iters=5)
v_joint, diag_joint = pipe_joint.apply(X2 * Y2**2, x2, kx2, y_grid=y2, ky=ky2)
exact_joint = X2 * Y2**2 + 2 * X2

# ==============================================================================
# VISUALIZATION
# ==============================================================================
fig = plt.figure(figsize=(18, 11))
fig.suptitle("Kohn-Nirenberg Subdomain Boundary Pipeline: Validation Suite", fontsize=16, fontweight='bold')

# 1D Plots (Top Row)
cases_1d = [
    (v_lap, exact_lap, geo_lap, r"1D Laplacian ($-\partial_x^2$) on $[-1, 1]$", diag_lap),
    (v_trans, exact_trans, geo_trans, r"1D Transport ($\partial_x$) on $[0, 2]$", diag_trans),
    (v_bes, None, geo_bes, r"1D Bessel Potential $(1+\xi^2)^{-1}$", diag_bes)
]

for i, (v, exact, geo, title, diag) in enumerate(cases_1d):
    ax = fig.add_subplot(2, 3, i+1)
    ax.plot(x, np.real(v), 'b-', lw=2, label='Pipeline $v_\Omega$')
    if exact is not None:
        ax.plot(x, np.real(exact), 'r--', lw=2, label='Exact')
    
    # Shade the subdomain Omega
    ax.fill_between(x, 0, geo.chi, color='gray', alpha=0.3, label=r'$\chi_\Omega$ (mask)')
    
    res_str = f"{abs(diag['residual_D']):.2e}" if diag['residual_D'] is not None else "N/A"
    ax.set_title(f"{title}\nResidual: {res_str} | Iters: {diag['n_iters']}")
    ax.legend(fontsize=9, loc='best')
    ax.set_xlabel('x')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0) # Keep plots tidy

# 2D Plots (Bottom Row)
cases_2d = [
    (v_disk, exact_disk, geo_disk, r"2D Laplacian on Disk", diag_disk),
    (v_ell, exact_ell, geo_ell, r"2D Anisotropic on Ellipse", diag_ell),
    (v_joint, exact_joint, geo_joint, r"2D Joint ($x\partial_x + \partial_y^2$)", diag_joint)
]

for i, (v, exact, geo, title, diag) in enumerate(cases_2d):
    ax = fig.add_subplot(2, 3, i+4)
    
    # Plot Absolute Error inside the domain
    err = np.abs(np.real(v) - np.real(exact)) * geo.chi
    im = ax.pcolormesh(X2, Y2, err, cmap='hot', shading='auto')
    plt.colorbar(im, ax=ax, label='Abs Error')
    
    # Overlay the exact boundary C
    ax.contour(X2, Y2, geo.g_func(X2, Y2), levels=[0], colors='cyan', linewidths=2, linestyles='--')
    
    res_str = f"{abs(diag['residual_D']):.2e}" if diag['residual_D'] is not None else "N/A"
    ax.set_title(f"{title}\nTrace Residual: {res_str}")
    ax.set_aspect('equal')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, exp, pi, sqrt


# Set seed for reproducibility
np.random.seed(42)

# Set up matplotlib figure layout
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
plt.subplots_adjust(hspace=0.35, wspace=0.3)

# ==============================================================================
# EXAMPLE 1 (1D): Fractional Laplacian (-Δ)^(0.7/2)
# ==============================================================================
x_sym, xi_sym = symbols('x xi', real=True)
alpha = 0.7
op_1d_frac = PseudoDifferentialOperator(abs(xi_sym)**alpha, [x_sym], mode='symbol')

N1 = 256
x1 = np.linspace(-10, 10, N1, endpoint=False)
dx1 = x1[1] - x1[0]
kx1 = 2 * np.pi * np.fft.fftfreq(N1, d=dx1)

u1 = np.exp(-x1**2) * np.cos(3 * x1)
v1 = op_1d_frac.apply(u1, x1, kx1, boundary_condition='periodic')

ax = axes[0, 0]
ax.plot(x1, u1.real, 'k--', label='Input $u(x)$')
ax.plot(x1, v1.real, 'r-', label=r'$(-\Delta)^{0.35} u(x)$')
ax.set_title('Ex 1: 1D Fractional Laplacian ($\\alpha=0.7$)')
ax.set_xlabel('$x$')
ax.set_ylabel('Amplitude')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# ==============================================================================
# EXAMPLE 2 (1D): Harmonic Oscillator -d^2/dx^2 + x^2
# ==============================================================================
op_1d_harm = PseudoDifferentialOperator(xi_sym**2 + x_sym**2, [x_sym], mode='symbol')

N2 = 256
x2 = np.linspace(-6, 6, N2, endpoint=False)
dx2 = x2[1] - x2[0]
kx2 = 2 * np.pi * np.fft.fftfreq(N2, d=dx2)

u2 = np.exp(-x2**2 / 2.0)
v2 = op_1d_harm.apply(u2, x2, kx2, boundary_condition='periodic')

ax = axes[0, 1]
ax.plot(x2, u2.real, 'k--', label=r'Ground State $e^{-x^2/2}$')
ax.plot(x2, v2.real, 'b-', label=r'$(-\partial_x^2 + x^2)u$')
ax.set_title('Ex 2: 1D Harmonic Oscillator')
ax.set_xlabel('$x$')
ax.set_ylabel('Amplitude')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# ==============================================================================
# EXAMPLE 3 (1D): Non-Periodic Transport Operator P = x * xi
# ==============================================================================
op_1d_trans = PseudoDifferentialOperator(x_sym * xi_sym, [x_sym], mode='symbol')

N3 = 128
x3 = np.linspace(-3, 3, N3)
dx3 = x3[1] - x3[0]
kx3 = 2 * np.pi * np.fft.fftfreq(N3, d=dx3)

u3 = np.exp(-4 * x3**2)
v3 = op_1d_trans.apply(u3, x3, kx3, boundary_condition='dirichlet')

ax = axes[0, 2]
ax.plot(x3, u3.real, 'k--', label='Input $u(x)$')
ax.plot(x3, v3.real, 'g-', label=r'$Op(x\xi)u$ (Dirichlet)')
ax.set_title(r'Ex 3: 1D Non-Periodic Transport $x\cdot D$')  # Fixed escape sequence warning with raw string 'r'
ax.set_xlabel('$x$')
ax.set_ylabel('Amplitude')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# ==============================================================================
# EXAMPLE 4 (2D): Fractional Schrödinger (-Δ)^(1/2) + V(x,y)
# ==============================================================================
y_sym, eta_sym = symbols('y eta', real=True)
symbol_2d_frac = sqrt(xi_sym**2 + eta_sym**2) + (x_sym**2 + 2 * y_sym**2)
op_2d_frac = PseudoDifferentialOperator(symbol_2d_frac, [x_sym, y_sym], mode='symbol')

N4 = 64
x4 = np.linspace(-4, 4, N4, endpoint=False)
y4 = np.linspace(-4, 4, N4, endpoint=False)
dx4, dy4 = x4[1] - x4[0], y4[1] - y4[0]
kx4 = 2 * np.pi * np.fft.fftfreq(N4, d=dx4)
ky4 = 2 * np.pi * np.fft.fftfreq(N4, d=dy4)

X4, Y4 = np.meshgrid(x4, y4, indexing='ij')
u4 = np.exp(-(X4**2 + Y4**2))
v4 = op_2d_frac.apply(u4, x4, kx4, y_grid=y4, ky=ky4, boundary_condition='periodic')

ax = axes[1, 0]
im = ax.contourf(X4, Y4, v4.real, levels=20, cmap='magma')
ax.set_title(r'Ex 4: 2D $(-\Delta)^{1/2} + (x^2 + 2y^2)$')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
fig.colorbar(im, ax=ax)

# ==============================================================================
# EXAMPLE 5 (2D): Anisotropic Variable-Coefficient Operator
# ==============================================================================
symbol_2d_var = (1 + x_sym**2) * xi_sym**2 + eta_sym**2
op_2d_var = PseudoDifferentialOperator(symbol_2d_var, [x_sym, y_sym], mode='symbol')

N5 = 64
x5 = np.linspace(-3, 3, N5, endpoint=False)
y5 = np.linspace(-3, 3, N5, endpoint=False)
dx5, dy5 = x5[1] - x5[0], y5[1] - y5[0]
kx5 = 2 * np.pi * np.fft.fftfreq(N5, d=dx5)
ky5 = 2 * np.pi * np.fft.fftfreq(N5, d=dy5)

X5, Y5 = np.meshgrid(x5, y5, indexing='ij')
u5 = np.sin(np.pi * X5 / 3) * np.cos(np.pi * Y5 / 3)
v5 = op_2d_var.apply(u5, x5, kx5, y_grid=y5, ky=ky5, boundary_condition='periodic')

ax = axes[1, 1]
im = ax.contourf(X5, Y5, v5.real, levels=20, cmap='viridis')
ax.set_title(r'Ex 5: 2D Variable $(1+x^2)\xi^2 + \eta^2$')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
fig.colorbar(im, ax=ax)

# ==============================================================================
# EXAMPLE 6 (2D): Subdomain Boundary Pipeline on a Circular Domain
# ==============================================================================
op_2d_lap = PseudoDifferentialOperator(xi_sym**2 + eta_sym**2 + 1, [x_sym, y_sym], mode='symbol')

N6 = 64
x6 = np.linspace(-2, 2, N6, endpoint=False)
y6 = np.linspace(-2, 2, N6, endpoint=False)
dx6, dy6 = x6[1] - x6[0], y6[1] - y6[0]
kx6 = 2 * np.pi * np.fft.fftfreq(N6, d=dx6)
ky6 = 2 * np.pi * np.fft.fftfreq(N6, d=dy6)

# Geometry definition: Circle of radius 1 centered at origin (g <= 0 inside domain)
g_circle = lambda xv, yv: xv**2 + yv**2 - 1.0

# FIX: Pass x6 and y6 explicitly as separate arguments to SubdomainGeometry
geo = SubdomainGeometry(x6, g_circle, dim=2, y_grid=y6)

pipe = SubdomainBoundaryPipeline(op_2d_lap, geo, f=1.0, max_defect_iters=6, defect_tol=1e-6)
u6_init = np.ones((N6, N6), dtype=complex)
v6, diag = pipe.apply(u6_init, x6, kx6, y_grid=y6, ky=ky6)

X6, Y6 = np.meshgrid(x6, y6, indexing='ij')
ax = axes[1, 2]
im = ax.contourf(X6, Y6, v6.real, levels=20, cmap='coolwarm')
# Plot subdomain boundary circle x^2 + y^2 = 1
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'r--', lw=2, label=r'Boundary $\partial\Omega$')
ax.set_title(r'Ex 6: Subdomain Solution ($-\Delta + I$)')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.legend(loc='upper right', fontsize=8)
fig.colorbar(im, ax=ax)

plt.suptitle("psiop_qwen Toolkit Demonstration: 3 1D and 3 2D Examples", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import sys

# Assuming these are imported from your updated psiop_qwen.py
from psiop_qwen import PseudoDifferentialOperator, SubdomainGeometry, SubdomainBoundaryPipeline

# Figure setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
fig.suptitle("Psiop Subdomain Integration (Pipeline Classes) — 1D & 2D Quality Benchmarks", fontsize=15, fontweight='bold')

def rel_l2_error(approx, exact, mask):
    norm_exact = np.linalg.norm(exact[mask])
    if norm_exact == 0: return np.linalg.norm(approx[mask])
    return np.linalg.norm((approx - exact)[mask]) / norm_exact

# Set up symbolic variables
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

# ==============================================================================
# 1D SUBDOMAIN EXAMPLES
# ==============================================================================
N1 = 256
x_grid1 = np.linspace(-10, 10, N1, endpoint=False)
dx1 = x_grid1[1] - x_grid1[0]
kx1 = 2 * np.pi * np.fft.fftfreq(N1, d=dx1)

u1 = np.sin(x_grid1)

# ------------------------------------------------------------------------------
# 1D Ex 1: Helmholtz Green's Filter 1/(1 + \xi^2) on Interval [-3, 3]
# ------------------------------------------------------------------------------
op_1d_1 = PseudoDifferentialOperator(1 / (1 + xi**2), [x], mode='symbol', apply_backend='direct')
geo1_1 = SubdomainGeometry(x_grid1, lambda xv: np.abs(xv) - 3.0, dim=1)
pipe1_1 = SubdomainBoundaryPipeline(op_1d_1, geo1_1, f=np.cos(x_grid1), max_defect_iters=8, defect_mode='gain')
v1_1, _ = pipe1_1.apply(u1, x_grid1, kx1)
v1_1 = v1_1.real

mask1_1 = geo1_1.chi > 0.5  
v1_1_exact = 0.5 * np.sin(x_grid1)
err1_1 = np.abs(v1_1 - v1_1_exact) * mask1_1
l2_1_1 = rel_l2_error(v1_1, v1_1_exact, mask1_1)

ax = axes[0, 0]
ax.plot(x_grid1, v1_1, 'r-', label=r'Subdomain Result $v_{\Omega}$')
ax.plot(x_grid1, v1_1_exact, 'k--', label='Exact Solution')
ax.plot(x_grid1, err1_1, 'm:', label=r'Pointwise Error (in $\Omega$)')
ax.axvspan(-3, 3, color='gray', alpha=0.15, label=r'Subdomain $\Omega$')
ax.set_title(r"1D Ex 1: Helmholtz Filter on Interval $[-3, 3]$")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend(loc='upper right')
ax.set_xlim(-6, 6)
ax.text(0.03, 0.05, f"Subdomain Rel $L_2$ Err: {l2_1_1:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# 1D Ex 2: Fractional Advection |\xi|^0.5 on Asymmetric Interval [-1, 4]
# ------------------------------------------------------------------------------
# NOTE: Bypass scope gate dynamically for this specific uncontrolled heuristic test.
psiop_mod = sys.modules[SubdomainBoundaryPipeline.__module__]
original_validate = psiop_mod._validate_pipeline_scope
psiop_mod._validate_pipeline_scope = lambda *args, **kwargs: True

op_1d_2 = PseudoDifferentialOperator(sp.Abs(xi)**0.5, [x], mode='symbol', apply_backend='direct')
geo1_2 = SubdomainGeometry(x_grid1, lambda xv: (xv - 1.5)**2 - 6.25, dim=1)
pipe1_2 = SubdomainBoundaryPipeline(op_1d_2, geo1_2, f=np.exp(-x_grid1**2), max_defect_iters=8, defect_mode='gain')
v1_2, _ = pipe1_2.apply(u1, x_grid1, kx1)
v1_2 = v1_2.real

psiop_mod._validate_pipeline_scope = original_validate # Restore scope gate

mask1_2 = geo1_2.chi > 0.5
v1_2_ref = op_1d_2.apply(u1, x_grid1, kx1, freq_window=None).real
err1_2 = np.abs(v1_2 - v1_2_ref) * mask1_2
l2_1_2 = rel_l2_error(v1_2, v1_2_ref, mask1_2)

ax = axes[1, 0]
ax.plot(x_grid1, v1_2, 'r-', label=r'Subdomain Result $v_{\Omega}$')
ax.plot(x_grid1, v1_2_ref, 'k--', label='Full-Space Reference')
ax.plot(x_grid1, err1_2, 'm:', label=r'Pointwise Error (in $\Omega$)')
ax.axvspan(-1, 4, color='gray', alpha=0.15, label=r'Subdomain $\Omega$')
ax.set_title(r"1D Ex 2: Fractional $|\xi|^{0.5}$ on Asymmetric $[-1, 4]$ (Out of Scope)")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend(loc='upper right')
ax.set_xlim(-5, 6)
ax.text(0.03, 0.05, f"Subdomain Rel $L_2$ Err: {l2_1_2:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# 1D Ex 3: Variable-Coefficient Diffusion on Disjoint Subdomains
# ------------------------------------------------------------------------------
expr_var_1d = (1 + 0.2 * sp.cos(x)) * xi**2
op_1d_3 = PseudoDifferentialOperator(expr_var_1d, [x], mode='symbol', apply_backend='direct')
geo1_3 = SubdomainGeometry(x_grid1, lambda xv: np.cos(np.pi * xv / 3.0), dim=1)
pipe1_3 = SubdomainBoundaryPipeline(op_1d_3, geo1_3, f=np.zeros_like(x_grid1), max_defect_iters=8, defect_mode='gain')

u1_gauss = np.exp(-0.5 * x_grid1**2)
v1_3, _ = pipe1_3.apply(u1_gauss, x_grid1, kx1)
v1_3 = v1_3.real

mask1_3 = geo1_3.chi > 0.5
d2u = (x_grid1**2 - 1) * u1_gauss
v1_3_exact = (1 + 0.2 * np.cos(x_grid1)) * (-d2u)
err1_3 = np.abs(v1_3 - v1_3_exact) * mask1_3
l2_1_3 = rel_l2_error(v1_3, v1_3_exact, mask1_3)

ax = axes[2, 0]
ax.plot(x_grid1, v1_3, 'r-', label=r'Subdomain Result $v_{\Omega}$')
ax.plot(x_grid1, v1_3_exact, 'k--', label='Exact Analytical')
ax.plot(x_grid1, err1_3, 'm:', label=r'Pointwise Error (in $\Omega$)')
ax.plot(x_grid1, mask1_3 * 0.5, 'c--', label=r'Mask $\chi_{\Omega}$')
ax.set_title("1D Ex 3: Variable Coeff. Operator on Disjoint Domains")
ax.set_xlabel("x")
ax.set_ylabel("Amplitude")
ax.legend(loc='upper right')
ax.set_xlim(-6, 6)
ax.text(0.03, 0.05, f"Subdomain Rel $L_2$ Err: {l2_1_3:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))


# ==============================================================================
# 2D SUBDOMAIN EXAMPLES
# ==============================================================================
N2 = 128
x_grid2 = np.linspace(-4, 4, N2, endpoint=False)
y_grid2 = np.linspace(-4, 4, N2, endpoint=False)
dx2, dy2 = x_grid2[1] - x_grid2[0], y_grid2[1] - y_grid2[0]

kx2 = 2 * np.pi * np.fft.fftfreq(N2, d=dx2)
ky2 = 2 * np.pi * np.fft.fftfreq(N2, d=dy2)

X2, Y2 = np.meshgrid(x_grid2, y_grid2, indexing='ij')
u2d = np.exp(-(X2**2 + Y2**2))

# ------------------------------------------------------------------------------
# 2D Ex 4: Circular Disk Subdomain for 2D Laplacian
# ------------------------------------------------------------------------------
op_2d_1 = PseudoDifferentialOperator(xi**2 + eta**2, [x, y], mode='symbol', apply_backend='direct')
geo2_1 = SubdomainGeometry(x_grid2, lambda X, Y: X**2 + Y**2 - 4.0, dim=2, y_grid=y_grid2)
pipe2_1 = SubdomainBoundaryPipeline(op_2d_1, geo2_1, f=np.zeros_like(X2), max_defect_iters=8, defect_mode='gain')
v2_1, _ = pipe2_1.apply(u2d, x_grid2, kx2, y_grid=y_grid2, ky=ky2)
v2_1 = v2_1.real

mask2_1 = geo2_1.chi > 0.5
v2_1_exact = 4 * (1 - (X2**2 + Y2**2)) * u2d
err2_1 = np.abs(v2_1 - v2_1_exact) * mask2_1
l2_2_1 = rel_l2_error(v2_1, v2_1_exact, mask2_1)

ax = axes[0, 1]
im1 = ax.imshow(err2_1.T, extent=[-4, 4, -4, 4], origin='lower', cmap='inferno')
ax.contour(X2, Y2, geo2_1.g_func(X2, Y2), levels=[0], colors='white', linewidths=1.5)
fig.colorbar(im1, ax=ax, label='Absolute Error')
ax.set_title(r"2D Ex 4: Laplacian Error on Disk Domain ($R=2$)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Disk Rel $L_2$ Err: {l2_2_1:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# 2D Ex 5: Elliptical Domain for Variable-Coefficient Operator
# ------------------------------------------------------------------------------
expr_2d_var = (1 + 0.3 * sp.cos(x)) * xi**2 + eta**2
op_2d_2 = PseudoDifferentialOperator(expr_2d_var, [x, y], mode='symbol', apply_backend='direct')
geo2_2 = SubdomainGeometry(x_grid2, lambda X, Y: (X / 2.5)**2 + (Y / 1.5)**2 - 1.0, dim=2, y_grid=y_grid2)
pipe2_2 = SubdomainBoundaryPipeline(op_2d_2, geo2_2, f=np.zeros_like(X2), max_defect_iters=8, defect_mode='gain')
v2_2, _ = pipe2_2.apply(u2d, x_grid2, kx2, y_grid=y_grid2, ky=ky2)
v2_2 = v2_2.real

mask2_2 = geo2_2.chi > 0.5
d2x = (4 * X2**2 - 2) * u2d
d2y = (4 * Y2**2 - 2) * u2d
v2_2_exact = (1 + 0.3 * np.cos(X2)) * (-d2x) + (-d2y)
err2_2 = np.abs(v2_2 - v2_2_exact) * mask2_2
l2_2_2 = rel_l2_error(v2_2, v2_2_exact, mask2_2)

ax = axes[1, 1]
im2 = ax.imshow(err2_2.T, extent=[-4, 4, -4, 4], origin='lower', cmap='inferno')
ax.contour(X2, Y2, geo2_2.g_func(X2, Y2), levels=[0], colors='white', linewidths=1.5)
fig.colorbar(im2, ax=ax, label='Absolute Error')
ax.set_title("2D Ex 5: Variable Operator Error on Ellipse")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Ellipse Rel $L_2$ Err: {l2_2_2:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ------------------------------------------------------------------------------
# 2D Ex 6: Square Box Subdomain for Directional Derivative
# ------------------------------------------------------------------------------
theta = np.pi / 3
op_2d_3 = PseudoDifferentialOperator(sp.I * (xi * np.cos(theta) + eta * np.sin(theta)), [x, y], mode='symbol', apply_backend='direct')
geo2_3 = SubdomainGeometry(x_grid2, lambda X, Y: np.maximum(np.abs(X) - 2.0, np.abs(Y) - 2.0), dim=2, y_grid=y_grid2)
pipe2_3 = SubdomainBoundaryPipeline(op_2d_3, geo2_3, f=np.zeros_like(X2), max_defect_iters=8, defect_mode='gain')
v2_3, _ = pipe2_3.apply(u2d, x_grid2, kx2, y_grid=y_grid2, ky=ky2)
v2_3 = v2_3.real

mask2_3 = geo2_3.chi > 0.5
v2_3_exact = -2 * (X2 * np.cos(theta) + Y2 * np.sin(theta)) * u2d
err2_3 = np.abs(v2_3 - v2_3_exact) * mask2_3
l2_2_3 = rel_l2_error(v2_3, v2_3_exact, mask2_3)

ax = axes[2, 1]
im3 = ax.imshow(err2_3.T, extent=[-4, 4, -4, 4], origin='lower', cmap='inferno')
ax.contour(X2, Y2, geo2_3.g_func(X2, Y2), levels=[0], colors='white', linewidths=1.5)
fig.colorbar(im3, ax=ax, label='Absolute Error')
ax.set_title(r"2D Ex 6: Directional Derivative Error on Box $[-2, 2]^2$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.text(0.03, 0.05, f"Box Rel $L_2$ Err: {l2_2_3:.2e}", transform=ax.transAxes, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()